In [249]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import lit, col
from dateutil.relativedelta import relativedelta
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import expr, col, round, avg, max, min, sum, count, when, lit
from itertools import product
from datetime import datetime
import pytz
import os
import logging

In [250]:

# Configura logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


In [251]:
def configure_spark():

    spark = (
        SparkSession.builder
        .appName("lakehouse-jupyter-book-pedidos")

        # conexão com cluster
        .master(os.getenv("SPARK_MASTER"))

        # 🚨 ESSENCIAL em Docker
        .config("spark.driver.host", os.getenv("SPARK_DRIVER_HOST"))
        .config("spark.driver.bindAddress", "0.0.0.0")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # S3 / MinIO
        .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT"))
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

        .config("spark.hadoop.fs.s3a.access.key", os.getenv("AWS_ACCESS_KEY_ID"))
        .config("spark.hadoop.fs.s3a.secret.key", os.getenv("AWS_SECRET_ACCESS_KEY"))

        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
        )

        # performance
        .config("spark.sql.adaptive.enabled", "true")

        # Delta fix
        .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")

        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    return spark

In [252]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [253]:
#executar todas as datas abaixo em (  data_exec_inicial  ) 
#201808, 201807, 201806, 201805, 201804, 201803, 201802, 201801, 201712, 201711, 201710, 201709
# converte YYYYMM -> date
data_exec_inicial = 201808
# converte YYYYMM -> date
data_dt = datetime.strptime(str(data_exec_inicial), "%Y%m")

# subtrai 12 meses
data_exec_final = int((data_dt - relativedelta(months=12)).strftime("%Y%m"))
data_exec_final

201708

In [254]:
spark = configure_spark()
spark

In [255]:
logging.info("Iniciando criação do book: pedidos")

df_orders = (
    spark.read.table("delta.`s3a://silver/olist_orders`")
)

df_orders.createOrReplaceTempView("df_orders")

2026-06-20 16:14:29,536 - INFO - Iniciando criação do book: pedidos


In [256]:
df_order_items = spark.read.table("delta.`s3a://silver/olist_order_items`")

df_order_items.createOrReplaceTempView("df_order_items")

In [257]:
df_orders_01 = spark.sql(f"""
    SELECT DISTINCT B.seller_id, A.safra 
    FROM df_orders AS A
    LEFT JOIN df_order_items AS B
    ON A.order_id = B.order_id
    WHERE A.safra = {data_exec_inicial}
""")

df_orders_01.createOrReplaceTempView("df_orders_01")

In [258]:
df_join = spark.sql(f"""
-- ============================================================
-- VARIÁVEIS PARA ANÁLISE DE CHURN DE SELLERS
-- ============================================================
-- PERÍODO: safra BETWEEN {data_exec_final} AND {data_exec_inicial}
-- ============================================================

WITH 
-- CTE 1: Agregação dos dados no nível do pedido
order_aggregated AS (
    SELECT 
        O.order_id,
        O.customer_id,
        O.order_status,
        O.order_purchase_timestamp,
        O.order_approved_at,
        O.order_delivered_carrier_date,
        O.order_delivered_customer_date,
        O.order_estimated_delivery_date,
        O.safra,
        OI.seller_id,
        OI.product_id,
        COUNT(OI.order_item_id) AS total_itens_pedido,
        SUM(OI.price) AS valor_total_pedido,
        SUM(OI.freight_value) AS frete_total_pedido
    FROM df_orders O
    LEFT JOIN df_order_items OI 
        ON O.order_id = OI.order_id
    WHERE OI.seller_id IS NOT NULL
        AND O.safra BETWEEN {data_exec_final} AND {data_exec_inicial}
    GROUP BY 
        O.order_id,
        O.customer_id,
        O.order_status,
        O.order_purchase_timestamp,
        O.order_approved_at,
        O.order_delivered_carrier_date,
        O.order_delivered_customer_date,
        O.order_estimated_delivery_date,
        O.safra,
        OI.seller_id,
        OI.product_id
),

-- CTE 2: Métricas por seller e safra
seller_metrics AS (
    SELECT 
        seller_id,
        safra,
        COUNT(DISTINCT order_id) AS qtdPedidos,
        SUM(total_itens_pedido) AS qtdItensVendidos,
        SUM(total_itens_pedido) AS qtdProdutosVendidos,
        COUNT(DISTINCT customer_id) AS qtdClientesAtendidos,
        SUM(valor_total_pedido) AS receitaTotal,
        CASE 
            WHEN COUNT(DISTINCT order_id) > 0 
            THEN SUM(valor_total_pedido) / COUNT(DISTINCT order_id) 
            ELSE 0 
        END AS ticketMedioPedido,
        CASE 
            WHEN SUM(total_itens_pedido) > 0 
            THEN SUM(valor_total_pedido) / SUM(total_itens_pedido) 
            ELSE 0 
        END AS ticketMedioItem,
        SUM(frete_total_pedido) AS valorFreteTotal,
        CASE 
            WHEN COUNT(DISTINCT order_id) > 0 
            THEN SUM(frete_total_pedido) / COUNT(DISTINCT order_id) 
            ELSE 0 
        END AS valorFreteMedio,
        COUNT(DISTINCT customer_id) AS qtdClientesUnicos,
        CASE 
            WHEN COUNT(DISTINCT customer_id) > 0 
            THEN COUNT(DISTINCT order_id) / COUNT(DISTINCT customer_id) 
            ELSE 0 
        END AS mediaPedidosPorCliente,
        COUNT(DISTINCT product_id) AS qtdProdutosDistintos,
        CASE 
            WHEN COUNT(DISTINCT product_id) > 0 
            THEN SUM(total_itens_pedido) / COUNT(DISTINCT product_id) 
            ELSE 0 
        END AS mediaItensPorProduto,
        CASE 
            WHEN COUNT(DISTINCT order_id) > 0 
            THEN COUNT(DISTINCT CASE WHEN order_status = 'CANCELED' THEN order_id END) / COUNT(DISTINCT order_id) 
            ELSE 0 
        END AS pctPedidoCancelado,
        CASE 
            WHEN COUNT(DISTINCT order_id) > 0 
            THEN COUNT(DISTINCT CASE WHEN order_status = 'DELIVERED' THEN order_id END) / COUNT(DISTINCT order_id) 
            ELSE 0 
        END AS pctPedidoEntregue,
        AVG(
            CASE 
                WHEN order_approved_at IS NOT NULL 
                    AND order_purchase_timestamp IS NOT NULL 
                THEN DATEDIFF(order_approved_at, order_purchase_timestamp)
                ELSE NULL 
            END
        ) AS tempoMedioAprovacaoPedido,
        AVG(
            CASE 
                WHEN order_delivered_carrier_date IS NOT NULL 
                    AND order_approved_at IS NOT NULL 
                THEN DATEDIFF(order_delivered_carrier_date, order_approved_at)
                ELSE NULL 
            END
        ) AS tempoMedioPostagem,
        AVG(
            CASE 
                WHEN order_delivered_customer_date IS NOT NULL 
                    AND order_purchase_timestamp IS NOT NULL 
                THEN DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)
                ELSE NULL 
            END
        ) AS tempoMedioEntrega,
        AVG(
            CASE 
                WHEN order_delivered_customer_date IS NOT NULL 
                    AND order_estimated_delivery_date IS NOT NULL 
                THEN DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date)
                ELSE NULL 
            END
        ) AS atrasoMedioEntrega,
        MAX(order_purchase_timestamp) AS ultimo_pedido_data,
        MAX(order_delivered_customer_date) AS ultima_entrega_data,
        SUM(valor_total_pedido) AS receita_total_base,
        COUNT(DISTINCT order_id) AS pedidos_total_base
    FROM order_aggregated
    GROUP BY seller_id, safra
),

-- CTE 3: Métricas de concentração por produto
concentracao_produto AS (
    SELECT 
        OI.seller_id,
        O.safra,
        OI.product_id,
        SUM(OI.price) AS receita_produto,
        COUNT(DISTINCT O.order_id) AS pedidos_produto,
        ROW_NUMBER() OVER (PARTITION BY OI.seller_id, O.safra ORDER BY SUM(OI.price) DESC) AS rank_receita_produto,
        ROW_NUMBER() OVER (PARTITION BY OI.seller_id, O.safra ORDER BY COUNT(DISTINCT O.order_id) DESC) AS rank_pedidos_produto
    FROM df_orders O
    LEFT JOIN df_order_items OI 
        ON O.order_id = OI.order_id
    WHERE OI.seller_id IS NOT NULL
        AND O.safra BETWEEN {data_exec_final} AND {data_exec_inicial}
    GROUP BY OI.seller_id, O.safra, OI.product_id
),

-- CTE 4: Métricas de concentração por cliente
concentracao_cliente AS (
    SELECT 
        OI.seller_id,
        O.safra,
        O.customer_id,
        SUM(OI.price) AS receita_cliente,
        COUNT(DISTINCT O.order_id) AS pedidos_cliente,
        ROW_NUMBER() OVER (PARTITION BY OI.seller_id, O.safra ORDER BY SUM(OI.price) DESC) AS rank_receita_cliente,
        ROW_NUMBER() OVER (PARTITION BY OI.seller_id, O.safra ORDER BY COUNT(DISTINCT O.order_id) DESC) AS rank_pedidos_cliente
    FROM df_orders O
    LEFT JOIN df_order_items OI 
        ON O.order_id = OI.order_id
    WHERE OI.seller_id IS NOT NULL
        AND O.safra BETWEEN {data_exec_final} AND {data_exec_inicial}
    GROUP BY OI.seller_id, O.safra, O.customer_id
),

-- CTE 5: Agregação das concentrações
concentracao_agregada AS (
    SELECT 
        seller_id,
        safra,
        MAX(CASE WHEN rank_receita_produto = 1 THEN receita_produto ELSE 0 END) AS receita_maior_produto,
        MAX(CASE WHEN rank_receita_cliente = 1 THEN receita_cliente ELSE 0 END) AS receita_maior_cliente,
        MAX(CASE WHEN rank_pedidos_cliente = 1 THEN pedidos_cliente ELSE 0 END) AS pedidos_maior_cliente,
        MAX(CASE WHEN rank_pedidos_produto = 1 THEN pedidos_produto ELSE 0 END) AS pedidos_maior_produto
    FROM (
        SELECT seller_id, safra, receita_produto, rank_receita_produto, 
               NULL AS receita_cliente, NULL AS rank_receita_cliente, 
               NULL AS pedidos_cliente, NULL AS rank_pedidos_cliente, 
               NULL AS pedidos_produto, NULL AS rank_pedidos_produto
        FROM concentracao_produto
        UNION ALL
        SELECT seller_id, safra, NULL, NULL, 
               receita_cliente, rank_receita_cliente, 
               pedidos_cliente, rank_pedidos_cliente, 
               NULL, NULL
        FROM concentracao_cliente
    ) t
    GROUP BY seller_id, safra
),

-- CTE 6: Junção final
final_metrics AS (
    SELECT 
        SM.seller_id,
        SM.safra,
        SM.qtdPedidos,
        SM.qtdItensVendidos,
        SM.qtdProdutosVendidos,
        SM.qtdClientesAtendidos,
        SM.receitaTotal,
        SM.ticketMedioPedido,
        SM.ticketMedioItem,
        SM.valorFreteTotal,
        SM.valorFreteMedio,
        SM.qtdClientesUnicos,
        SM.mediaPedidosPorCliente,
        SM.qtdProdutosDistintos,
        SM.mediaItensPorProduto,
        SM.pctPedidoCancelado,
        SM.pctPedidoEntregue,
        SM.tempoMedioAprovacaoPedido,
        SM.tempoMedioPostagem,
        SM.tempoMedioEntrega,
        SM.atrasoMedioEntrega,
        CASE 
            WHEN SM.receita_total_base > 0 
            THEN CA.receita_maior_produto / SM.receita_total_base 
            ELSE 0 
        END AS pctReceitaMaiorProduto,
        CASE 
            WHEN SM.receita_total_base > 0 
            THEN CA.receita_maior_cliente / SM.receita_total_base 
            ELSE 0 
        END AS pctReceitaMaiorCliente,
        CASE 
            WHEN SM.pedidos_total_base > 0 
            THEN CA.pedidos_maior_cliente / SM.pedidos_total_base 
            ELSE 0 
        END AS pctPedidosMaiorCliente,
        CASE 
            WHEN SM.pedidos_total_base > 0 
            THEN CA.pedidos_maior_produto / SM.pedidos_total_base 
            ELSE 0 
        END AS pctPedidosMaiorProduto
    FROM seller_metrics SM
    LEFT JOIN concentracao_agregada CA
        ON SM.seller_id = CA.seller_id 
        AND SM.safra = CA.safra
)

-- SELECT FINAL
SELECT 
    seller_id,
    safra,
    qtdPedidos,
    qtdItensVendidos,
    qtdProdutosVendidos,
    qtdClientesAtendidos,
    receitaTotal,
    ticketMedioPedido,
    ticketMedioItem,
    valorFreteTotal,
    valorFreteMedio,
    qtdClientesUnicos,
    mediaPedidosPorCliente,
    qtdProdutosDistintos,
    mediaItensPorProduto,
    pctPedidoCancelado,
    pctPedidoEntregue,
    tempoMedioAprovacaoPedido,
    tempoMedioPostagem,
    tempoMedioEntrega,
    atrasoMedioEntrega,
    pctReceitaMaiorProduto,
    pctReceitaMaiorCliente,
    pctPedidosMaiorCliente,
    pctPedidosMaiorProduto
FROM final_metrics
ORDER BY seller_id, safra;
""")

df_join.createOrReplaceTempView("df_join")

In [259]:
df_join.createOrReplaceTempView("df_transacoes")

In [260]:
df_temp_01 = spark.sql("""
WITH base AS (
    SELECT
        *,
        TO_DATE(CONCAT(safra, '01'), 'yyyyMMdd') AS data_dt
    FROM df_transacoes
)

SELECT
    *,
    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY seller_id), -1)
             AND MAX(data_dt) OVER (PARTITION BY seller_id)
        THEN 1 ELSE 0
    END AS u1m,

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY seller_id), -3)
             AND MAX(data_dt) OVER (PARTITION BY seller_id)
        THEN 1 ELSE 0
    END AS u3m,


    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY seller_id), -6)
             AND MAX(data_dt) OVER (PARTITION BY seller_id)
        THEN 1 ELSE 0
    END AS u6m,

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY seller_id), -9)
             AND MAX(data_dt) OVER (PARTITION BY seller_id)
        THEN 1 ELSE 0
    END AS u9m,    

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY seller_id), -12)
             AND MAX(data_dt) OVER (PARTITION BY seller_id)
        THEN 1 ELSE 0
    END AS u12m

FROM base
ORDER BY seller_id, safra
""")

df_temp_01.createOrReplaceTempView("df_temp_01")

In [261]:
# Definição das variáveis
coluna_chave = "seller_id"
colunas_flags = ['u1m', 'u3m', 'u6m', 'u9m', 'u12m']

# Lista de colunas de valores
colunas_valores = [
    'qtdPedidos', 'qtdItensVendidos', 'qtdProdutosVendidos', 'qtdClientesAtendidos', 'receitaTotal', 'ticketMedioPedido', 'ticketMedioItem', 'valorFreteTotal', 'valorFreteMedio', 'qtdClientesUnicos',
    'mediaPedidosPorCliente', 'qtdProdutosDistintos', 'mediaItensPorProduto', 'pctPedidoCancelado' , 'pctPedidoEntregue', 'tempoMedioAprovacaoPedido', 'tempoMedioPostagem'
    , 'tempoMedioEntrega', 'atrasoMedioEntrega', 'pctReceitaMaiorProduto', 'pctReceitaMaiorCliente', 'pctPedidosMaiorCliente', 'pctPedidosMaiorProduto'
]

# Configuração dos indicadores
"""indicadores_config = {
    'IND_PDD': {'alias': 'PDD', 'valores': ['S']},
    'IND_WO': {'alias': 'WO', 'valores': ['R']},
    'IND_PCCR': {'alias': 'PCCR', 'valores': ['W']},
    'IND_ACA': {'alias': 'ACA', 'valores': ['N']},
    'IND_PRIMEIRA_FAT': {'alias': 'PRIM_FAT', 'valores': ['S']},
    'IND_FRAUDE': {'alias': 'FRAUDE', 'valores': ['N']}
}"""



def gerar_sql_dinamico():
    #selects = ["seller_id", "seller_region"]
    selects = ["seller_id"]
    
    # Agregações básicas
    for flag in colunas_flags:
        for valor in colunas_valores:
            selects.append(f"round(avg(case when {flag} = 1 then {valor} else NULL end), 2) as vl_med_{flag}_{valor}_pedidos")
            selects.append(f"round(max(case when {flag} = 1 then {valor} else NULL end), 2) as vl_max_{flag}_{valor}_pedidos")
            selects.append(f"round(min(case when {flag} = 1 then {valor} else NULL end), 2) as vl_min_{flag}_{valor}_pedidos")
            selects.append(f"round(stddev(case when {flag} = 1 then {valor} else NULL end), 2) as vl_std_{flag}_{valor}_pedidos")
            selects.append(f"round(count(case when {flag} = 1 then {valor} else NULL end), 2) as vl_qtd_{flag}_{valor}_pedidos")
    # Agregações com indicadores
    """for indicador, info in indicadores_config.items():
        for valor_indicador in info['valores']:
            for flag in colunas_flags:
                for valor in colunas_valores:
                    alias = info['alias']
                    selects.append(f"round(avg(case when {flag} = 1 and {indicador} = '{valor_indicador}' then {valor} else NULL end), 2) as vl_med_{flag}_{alias}_{valor_indicador}_{valor}_orders")
                    #selects.append(f"round(max(case when {flag} = 1 and {indicador} = '{valor_indicador}' then {valor} else NULL end), 2) as vl_max_{flag}_{alias}_{valor_indicador}_{valor}_orders")
                    #selects.append(f"round(count(case when {flag} = 1 and {indicador} = '{valor_indicador}' then {valor} else NULL end), 2) as vl_qtd_{flag}_{alias}_{valor_indicador}_{valor}_orders")"""
    
    sql_query = f"""
    SELECT
        {', '.join(selects)}
    FROM df_temp_01
    GROUP BY seller_id
    ORDER BY seller_id
    """
    
    return sql_query

# Executar SQL dinâmico
sql_dinamico = gerar_sql_dinamico()

df_temp_02 = spark.sql(sql_dinamico)

df_temp_02.createOrReplaceTempView("df_temp_02")
print(f"Shape: {df_temp_02.count()} linhas, {len(df_temp_02.columns)} colunas")

Shape: 2741 linhas, 576 colunas


In [262]:
def add_temporal_ratio_columns(
    df,
    base_prefix="vl_med",
    ratio_prefix="raz_med",
    windows=("u1m", "u3m", "u6m", "u9m", "u12m"),
    suffix="_pedidos"
):
    """
    Função que mantém todas as colunas originais do DataFrame
    e adiciona novas colunas de razão temporal entre janelas consecutivas.

    Exemplo de razão criada:
    RAZ_MED_U1M_U3M_FAT_ATRASO = VL_MED_U1M_FAT_ATRASO / VL_MED_U3M_FAT_ATRASO
    """

    # Cria uma lista com todas as colunas originais do DataFrame
    # Isso garante que nenhuma coluna existente será removida
    base_cols = [F.col(c) for c in df.columns]

    # Lista que armazenará as expressões das novas colunas de razão
    ratio_exprs = []

    # Conjunto com os nomes das colunas do DataFrame
    # Usado para checar rapidamente se uma coluna existe
    df_cols = set(df.columns)

    # Gera pares de janelas consecutivas
    # Exemplo: (U1M, U3M), (U3M, U6M), ...
    window_pairs = list(zip(windows[:-1], windows[1:]))

    # Percorre cada par de janelas (numerador e denominador)
    for num_win, den_win in window_pairs:

        # Percorre todas as colunas do DataFrame
        for col_num in df.columns:

            # Verifica se a coluna pertence à janela do numerador
            # e segue o padrão: VL_MED__
            if not col_num.startswith(f"{base_prefix}_{num_win}_"):
                continue

            # Deriva o nome da coluna do denominador
            # Substitui a janela do numerador pela janela do denominador
            col_den = col_num.replace(
                f"{base_prefix}_{num_win}_",
                f"{base_prefix}_{den_win}_"
            )

            # Se a coluna do denominador não existir, ignora
            if col_den not in df_cols:
                continue

            # Extrai o nome da feature base
            # Remove prefixo (VL_MED__) e o sufixo (_ATRASO)
            feature = (
                col_num
                .replace(f"{base_prefix}_{num_win}_", "")
                .replace(suffix, "")
            )

            # Define o nome da nova coluna de razão temporal
            # Exemplo: RAZ_MED_U1M_U3M_FAT_ATRASO
            ratio_name = (
                f"{ratio_prefix}_{num_win}_{den_win}_"
                f"{feature}{suffix}"
            )

            # Cria a expressão da razão temporal
            # Realiza a divisão apenas quando o denominador é diferente de zero
            # Caso contrário, retorna NULL
            ratio_exprs.append(
                F.when(
                    F.col(col_den) != 0,
                    F.col(col_num) / F.col(col_den)
                ).alias(ratio_name)
            )

    # Retorna o DataFrame mantendo todas as colunas originais
    # e adicionando as novas colunas de razão temporal
    return df.select(*base_cols, *ratio_exprs)


# Aplica a função ao DataFrame anterior, gerando um novo DataFrame enriquecido
df_temp_03 = add_temporal_ratio_columns(df_temp_02)


df_temp_03.createOrReplaceTempView("df_temp_03")

print(f"Shape: {df_temp_03.count()} linhas, {len(df_temp_03.columns)} colunas")

Shape: 2741 linhas, 668 colunas


In [263]:
df_temp_04 = df_orders_01.alias("t1") \
    .join(df_temp_03.alias("t2"), "seller_id", "left") \
    .withColumn("safra", lit(data_exec_inicial)) \
    .withColumn("datproc", lit(dthproc)) \

df_temp_04.createOrReplaceTempView("df_temp_04")
df_temp_04.count()

1264

In [264]:
spark.sql("""
-- 4) Calcula hash para idempotência (evita UPDATE sem mudança real)
CREATE OR REPLACE TEMP VIEW stage_orders_final AS
SELECT
  *,
  sha2(concat_ws('||',
    coalesce(seller_id,''),
    coalesce(cast(safra as int),''),
    coalesce(cast(vl_med_u1m_qtdPedidos_pedidos as int),''),
    coalesce(cast(vl_med_u1m_receitaTotal_pedidos as int),'')
    --coalesce(product_id,''),
    --coalesce(seller_id,''),
    --coalesce(date_format(shipping_limit_date,'yyyy-MM-dd'),''),
    --coalesce(cast(price as string),''),
    --coalesce(cast(freight_value as string),'')
  ), 256) AS row_hash
FROM df_temp_04;
""")


DataFrame[]

In [265]:
spark.sql("""
        CREATE DATABASE IF NOT EXISTS gold
        LOCATION 's3a://gold'
""")

DataFrame[]

In [266]:
from delta.tables import DeltaTable

path = "s3a://gold/book_pedidos"

if not spark.catalog.tableExists("gold.book_pedidos"):
    
    # Se já existe Delta no storage → só registra
    if DeltaTable.isDeltaTable(spark, path):
        spark.sql(f"""
            CREATE TABLE gold.book_pedidos
            USING DELTA
            LOCATION '{path}'
        """)
    
    # Se não existe nada → cria do zero
    else:
        spark.sql(f"""
            CREATE TABLE gold.book_pedidos
            USING DELTA
            PARTITIONED BY (safra)
            LOCATION '{path}'
            AS SELECT * FROM stage_orders_final WHERE 1=0
        """)


In [267]:
spark.sql("""
MERGE INTO gold.book_pedidos AS t
USING stage_orders_final AS s
ON t.seller_id = s.seller_id
AND t.safra = s.safra

WHEN MATCHED AND (t.row_hash IS NULL OR t.row_hash <> s.row_hash) THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *
;
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [268]:
logging.info("Finalizando criação do book: pedidos")
spark.stop()

2026-06-20 16:15:22,664 - INFO - Finalizando criação do book: pedidos
